## Delta Lake

What is the Delta Lake?
- Open-source storage framework created by Databricks
- Built on top of parquet
- Native integration with Apache Spark

Features
- ACID transactions
- Schema enforcement and evolution
- Data versioning (time travel)
- Unified Batch Streaming
- Scalable metadata handling
- Combines the best features of data lakes and data lakehouses, => Delta Lake


### What Are ACID Transactions?

ACID stands for Atomicity, Consistency, Isolation, and Durability. These properties ensure reliable database transactions:
- **Atomicity**: Each transaction is all-or-nothing.
- **Consistency**: Data remains valid after a transaction.
- **Isolation**: Transactions do not interfere with each other.
- **Durability**: Completed transactions persist even after failures.

How Delta Works
- Each _insert_, _update_, and _delete_ computes a _Delta_ from the current version
- Commits a 'new' version to the _delta_log
- Each Select gets manifest data from the _delta_log
- Each Select returns the amalgam of the files in the version

![](/Volumes/workspace/pyspark_learning/raw_files/images/delta.png)

Delta Lake is the default for Databricks

read:  ```spark.read.format('delta').load('table path')```

write: ```spark.write.format('delta').mode('append').save('table path')```

Since is default you can simply omit format if delta is desired:
```spark.write.mode('append').save('table path)```

Commands for inspection of delta

```DESCRIBE HISTORY```

```DESCRIBE DETAIL```



## Optimization and Performance

Key Performance Factors
- Resource utilization (CPU, Mem, Disk)
- Data Characteristics (size, format, distribution)
- Configuration and cluster setup


Common Bottlenecks

- Data skew and uneven partitioning
- Excessive shuffling
- Memory and GC


### What is Shuffling in Databricks?

Shuffling is the process of redistributing data across partitions in a cluster, typically during operations like joins, aggregations, or groupBy. It involves moving data between nodes, which can impact performance due to network and disk I/O.

## Spark Partitioning

Partioning is the foundation of distributed processing performance

- DataFrame partitioning determines data distribution
- Partitions are determined by:
  - Files or blocks when data is read from a table or a directory
  - As a result of Wide Transformations:  _groupBy()_, _join()_ or _repartition()/coalesce()_
    - These are referred to as shuffle partitions


The importance of data distribution
- Enables parallel processing across executors
- Impacts memory per executor
- Influences join and aggregation efficiency
- Determines network traffic patterns

## Spark Application Execution

Driver --> Job(s) --> Stage(s) --> Task(s)


Above logical actions correspond the following entities:

(Driver/ClusterManager/Workers/Executors/Tasks)

Tasks execute in parallel across worker nodes

![](/Volumes/workspace/pyspark_learning/raw_files/images/spark_execution.png)

Spark queries are broken down into logical plans.  The plan in is then  opitimized (Catalyst Optimizer) to reduce redundant code and for object reuse.  Then, multiple physical plans are created as execution options.  The lowest cost physical plan is selected and then executed (Tungsten)

![](/Volumes/workspace/pyspark_learning/raw_files/images/plan.png)

### Lazy Evaluation

- Data transformations are collected and recorded in a plan, and not acted upon until an execution is called.  Example ```display(), count(), write(), collect()```, etc.

- Actions are methods on dataframes which produce a result.  A trsansformation produces another dataframe or data object

- Delaying the execution allows for spark to create an optimized plan which includes both transformations and actions based on those transformations

In [0]:

df = spark.read.table('workspace.pyspark_learning.countries_consolidated'). \
    groupBy('region').sum('population')



### Explain Plan

In [0]:
"""
the following, without options passed to the method will explain the physical plan behind creating the dataframe
"""
df.explain()

In [0]:
"""
Physical plan with explanations
"""
df.explain(mode='formatted')

In [0]:
"""
Physical and logical plan
"""
df.explain(mode='extended')

### Query Performance

In [0]:
"""
When using serverless compute, with the action/execution below running, one can click on the
'See performance', then click
 'df.display()' link under the 'Statement' heading to see query statistics. Then,
 click on 'query profile' to see the query stages (which stage does what rows and memmory-wise)
"""

df.display()

### Caching
There are two caches in DBX, disk-cache and spark cache (memnory)
The recommendation is to use the DBX disk-cache which is enabled by default

### Data Shuffling

- Shuffling is when data is moved around between your nodes or executors during execution causing additional time for your queries due to network I/O overhead.
- Wide transformations such as  _groupBy()_,  _distinct()_, _orderBy()_, and _joins()_ cause shuffle.  This where data needs to be moved between executors (which exist on workers)
- A shuffle is a process which compares data across partitions
- If the subset of data is small enough, it can be _broadcast_ to all executors thus eliminating a more expensive shuffle operation.  This is called _broadcasting_
- Narrow transformations like _filter()_ do not cause suffling


**Mitigate Suffling By**

- Use fewer, more powerful worker nodes
- Filter data as much as possible BEFORE using any shuffle operations such as groupBy(), distinct(), orderBy(), and joins()
- Column pruning (reduce the number of columns in the df)
- Denormalize data (strategic process of reintroducing redundancy into a normalized database to boost read performance, typically by combining tables or adding calculated fields)
- Partition data on the columns that trigger suffles

### Broadcast Joins

Data tables smaller than 10mb are Broadcast to all executors

example of broadcast function.  Below df2 is being broadcast
```
df1.join(broadcast(df2), 'itemID')
```

### Partitions
- Partitions are only of benefit if the data is **greater** than 1 GB
- Partitions should be greater than 1 GB but less than 1 TB in size
- No one partion should dwarf the rest, known as 'data skew'
- Manual partitioning is not recommended as its normally not used correctly, there are better options

### Data Skipping
- Performance optimization technique that prevents DBX from reading uncessary files during queries.
- Transaction logs contain stats on delta formatted files, such as min and max values for columns contained in files.  

If, for example, a query is executed where a value is required that is larger or smaller than the max or min recorded for a specific file, that file will be _skipped_ and not read.  This reduces processing time as a file is not read for data which does not match the query conditions

### Z-Ordering

Z-Ordering is an optimization technique which can be used on Detla files and Tables.

It is a data clustering technique which co-locates related information in the same set of files on storage enabling queries to execute faster

This co-locality is automatically used by Delta Lake on databricks data-skipping algorithms to reduce the amount of data needed to be read to to satisfy query conditions.

In the example below, 4 files is reduced to two files via z-ordering similar populations together.  Depending on the size pppulation queried, only one file need be scanned for data
![](/Volumes/workspace/pyspark_learning/raw_files/images/z-order.png)


Syntax

```OPTIMIZE catalog.schema.table ZORDER BY (column(s));```

```OPTINIZE delta.`<file path>` ZORDER BY (column(s));```

This is run post-table creation and needs to be run periodically as a maintenance task


### Liquid Clustering

This is a more modern and efficient version of z-order which runs automatically in the background.

- can be enabled during table creation or after table creation
- this is NOT compatible with manual z-order or manual partioning.  To use Liquid clustering, you must let DBX manage everything for you
- to enable, add the ```CLUSTER BY``` phrase to table creation statement


```
df = spark.read.table('table1')
df.writeTo('table1').using('delta').cluserBy('col0').create()
```

You can let DBX decide how to cluster for you by using the the ```clusterByAuto``` option.

```
df = spark.read.table('table1')
df.writeTo('table1').using('delta').option('clusterByAuto', 'true').create()
```


### Spark Configurations
This is using ```spark.conf.set``` to alter the configuration on a cluster.  This is NOT available on Severless compute.

I have used this to set a variable sharing it between cell types, passing a python variable to a SQL query.

To see all configs set, ```spark.conf.getAll```

### Repartition

- Repartition is a dataframe method which is used to change the number of partions in a dataframe
- Can be used to increase or decrease numbers of partitions
- Use when you don't know if the command you are issuing will increase or decrease the number of partitions
- full shuffle is executed to partition data evenly

Syntax exmples


Repartition to 10 partitions
```
# I don't know how many partitions there are currently, but I know I want 10
df.repartition(10)
```

Specify columm to partion on
```
df.repartition("age")
```

Specify number of partitions and columns
```
df.repartition(7, "age", "name")
```


### Coalesce

- Does not shuffle, merges existing partitions
- Decreases partition numbers only
- Use only if you know the command will decrease the number of partitions, else error
- May create skewed, uneven partitions
- Faster and less resource intensive than `repartition`


examples

Reduce to 10 partitions
```
df.coalesce(10)
```

### Spark Storage Levels

example
```
from pyspark import StorageLevel
df.persist(StorageLevel=DISK_ONLY)
```

- MEMORY_ONLY: Stores data as deserialized objects in JVM. Fast but can cause OOM (Out of Memory) errors.
- MEMORY_AND_DISK: Stores data in memory; spills excess to disk.
- MEMORY_ONLY_SER / MEMORY_AND_DISK_SER: Stores data in serialized format (smaller footprint, higher CPU usage for serialization).
- DISK_ONLY: Stores data entirely on disk.